In [91]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch_mlir import fx

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))
from tutorial._infra import torch_frontend as torch_nb

class GemmModule(nn.Module):
    def forward(self, inputs):
        A, B = inputs
        return A @ B

m = GemmModule().eval()

A = torch.randn(1024, 2048, dtype=torch.float32)
B = torch.randn(2048, 512, dtype=torch.float32)
example_input = (A, B)

tm = fx.export_and_import(m, example_input, func_name="kernel")
torch_ir = tm.operation.get_asm()

out_file = torch_nb.ARTIFACTS_DIR / "gemm_torch.mlir"
out_file.write_text(torch_ir)
print(f"Wrote Torch dialect IR → {out_file.resolve()}")
print(torch_ir)


Wrote Torch dialect IR → /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/gemm_torch.mlir
module {
  func.func @kernel(%arg0: !torch.vtensor<[1024,2048],f32>, %arg1: !torch.vtensor<[2048,512],f32>) -> !torch.vtensor<[1024,512],f32> {
    %0 = torch.aten.mm %arg0, %arg1 : !torch.vtensor<[1024,2048],f32>, !torch.vtensor<[2048,512],f32> -> !torch.vtensor<[1024,512],f32>
    return %0 : !torch.vtensor<[1024,512],f32>
  }
}



In [92]:
from tutorial._infra import torch_frontend as torch_nb

pipeline = (
    "builtin.module("
    "torch-function-to-torch-backend-pipeline,"
    "torch-backend-to-linalg-on-tensors-backend-pipeline,"
    "torch-verify-linalg-on-tensors-backend-contract"
    ")"
)

torch_ir = torch_nb.ARTIFACTS_DIR / "gemm_torch.mlir"
linalg_ir = torch_nb.ARTIFACTS_DIR / "gemm_linalg.mlir"

torch_nb.run(
    [
        torch_nb.torch_mlir_opt,
        torch_ir,
        f"-pass-pipeline={pipeline}",
        "-o",
        linalg_ir,
    ]
)

print(f"Wrote Linalg IR → {linalg_ir}")
print(linalg_ir.read_text())

Wrote Linalg IR → /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/gemm_linalg.mlir
module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %0 = tensor.empty() : tensor<1024x512xf32>
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<1024x512xf32>) -> tensor<1024x512xf32>
    %2 = linalg.matmul ins(%arg0, %arg1 : tensor<1024x2048xf32>, tensor<2048x512xf32>) outs(%1 : tensor<1024x512xf32>) -> tensor<1024x512xf32>
    return %2 : tensor<1024x512xf32>
  }
}




In [93]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-cleanup-linalg,im2col-to-matmul)"
    ")"
)

gemm_linalg_im2col_clean = torch_nb.ARTIFACTS_DIR / "gemm_linalg_im2col_clean.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        linalg_ir,  # from the previous cell
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_linalg_im2col_clean,
    ]
)

print(gemm_linalg_im2col_clean.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %0 = tensor.empty() : tensor<1024x512xf32>
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<1024x512xf32>) -> tensor<1024x512xf32>
    %2 = linalg.matmul ins(%arg0, %arg1 : tensor<1024x2048xf32>, tensor<2048x512xf32>) outs(%1 : tensor<1024x512xf32>) -> tensor<1024x512xf32>
    return %2 : tensor<1024x512xf32>
  }
}




In [94]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(convert-linalg-to-cinm)"
    ")"
)

gemm_cinm0 = torch_nb.ARTIFACTS_DIR / "gemm_cinm0.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_linalg_im2col_clean,  # from the previous cell
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm0,
    ]
)

print(gemm_cinm0.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %0 = tensor.empty() : tensor<1024x512xf32>
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<1024x512xf32>) -> tensor<1024x512xf32>
    %2 = cinm.op.gemm %arg0, %arg1 : (tensor<1024x2048xf32>, tensor<2048x512xf32>) -> tensor<1024x512xf32>
    return %2 : tensor<1024x512xf32>
  }
}




In [95]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemm tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemv tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemm tile-sizes=32x32x32x32},"
    "cinm-annotate-tiles{ops=activate tile-sizes=32}"
    ")"
)

gemm_cinm1 = torch_nb.ARTIFACTS_DIR / "gemm_cinm1.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm0,  # from the previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm1,
    ]
)

print(gemm_cinm1.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %0 = tensor.empty() : tensor<1024x512xf32>
    %1 = linalg.fill ins(%cst : f32) outs(%0 : tensor<1024x512xf32>) -> tensor<1024x512xf32>
    %2 = cinm.compute attributes {tileSizes = array<i64: 32, 32, 32>} -> tensor<1024x512xf32> {
      %3 = cinm.op.gemm %arg0, %arg1 : (tensor<1024x2048xf32>, tensor<2048x512xf32>) -> tensor<1024x512xf32>
      cinm.yield %3 : tensor<1024x512xf32>
    }
    return %2 : tensor<1024x512xf32>
  }
}




In [96]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-gemm-to-gemv{split-dim=2}"
    ")"
)

gemm_cinm2 = torch_nb.ARTIFACTS_DIR / "gemm_cinm2.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm1,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm2,
    ]
)

print(gemm_cinm2.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %c0 = arith.constant 0 : index
    %0 = cinm.compute attributes {tileSizes = array<i64: 32, 32>} -> tensor<1024x512xf32> {
      %1 = tensor.empty() : tensor<1024x512xf32>
      %2 = affine.for %arg2 = 0 to 512 iter_args(%arg3 = %1) -> (tensor<1024x512xf32>) {
        %extracted_slice = tensor.extract_slice %arg1[%c0, %arg2] [2048, 1] [1, 1] : tensor<2048x512xf32> to tensor<2048x1xf32>
        %collapsed = tensor.collapse_shape %extracted_slice [[0, 1]] : tensor<2048x1xf32> into tensor<2048xf32>
        %3 = cinm.op.gemv %arg0, %collapsed : (tensor<1024x2048xf32>, tensor<2048xf32>) -> tensor<1024xf32>
        %expanded = tensor.expand_shape %3 [[0, 1]] output_shape [1024, 1] : tensor<1024xf32> into tensor<1024x1xf32>
        %inserted_slice = tensor.insert_slice %expanded into %arg3[%c0, %arg2] [1024, 1] [1, 1] : tensor<1024x1xf32> into tensor<1024x512xf32>
        affi

In [97]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-tiling"
    ")"
)

gemm_cinm3 = torch_nb.ARTIFACTS_DIR / "gemm_cinm3.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm2,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm3,
    ]
)

print(gemm_cinm3.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %c0 = arith.constant 0 : index
    %0 = cinm.compute attributes {tileSizes = array<i64: 32, 32>} -> tensor<1024x512xf32> {
      %1 = tensor.empty() : tensor<1024x512xf32>
      %2 = affine.for %arg2 = 0 to 512 iter_args(%arg3 = %1) -> (tensor<1024x512xf32>) {
        %extracted_slice = tensor.extract_slice %arg1[%c0, %arg2] [2048, 1] [1, 1] : tensor<2048x512xf32> to tensor<2048x1xf32>
        %collapsed = tensor.collapse_shape %extracted_slice [[0, 1]] : tensor<2048x1xf32> into tensor<2048xf32>
        %3 = tensor.empty() : tensor<1024xf32>
        %c1024 = arith.constant 1024 : index
        %c2048 = arith.constant 2048 : index
        %c32 = arith.constant 32 : index
        %c32_0 = arith.constant 32 : index
        %c0_1 = arith.constant 0 : index
        %c1024_2 = arith.constant 1024 : index
        %c32_3 = arith.constant 32 : index
        %4 = scf.for %arg4 = 

In [98]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "lower-affine,scf-for-loop-canonicalization,"
    "func.func(cinm-gemv-min-write)"
    ")"
)

gemm_cinm4 = torch_nb.ARTIFACTS_DIR / "gemm_cinm4.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm3,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm4,
    ]
)

print(gemm_cinm4.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %0 = cinm.compute attributes {tileSizes = array<i64: 32, 32>} -> tensor<1024x512xf32> {
      %1 = tensor.empty() : tensor<1024x512xf32>
      %2 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %1) -> (tensor<1024x512xf32>) {
        %3 = arith.subi %c1024, %arg2 : index
        %4 = arith.cmpi ugt, %3, %c32 : index
        %5 = arith.select %4, %c32, %3 : index
        %cst_0 = arith.constant 0.000000e+00 : f32
        %6 = tensor.empty(%5) : tensor<?x512xf32>
        %7 = linalg.fill ins(%cst_0 : f32) outs(%6 : tensor<?x512xf32>) -> tensor<?x512xf32>
        %8 = scf.for %arg4 = %c0 to %c

In [99]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-decompose-accum)"
    ")"
)

gemm_cinm5 = torch_nb.ARTIFACTS_DIR / "gemm_cinm5.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm4,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm5,
    ]
)

print(gemm_cinm5.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %0 = cinm.compute attributes {tileSizes = array<i64: 32, 32>} -> tensor<1024x512xf32> {
      %1 = tensor.empty() : tensor<1024x512xf32>
      %2 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %1) -> (tensor<1024x512xf32>) {
        %3 = arith.subi %c1024, %arg2 : index
        %4 = arith.cmpi ugt, %3, %c32 : index
        %5 = arith.select %4, %c32, %3 : index
        %cst_0 = arith.constant 0.000000e+00 : f32
        %6 = tensor.empty(%5) : tensor<?x512xf32>
        %7 = linalg.fill ins(%cst_0 : f32) outs(%6 : tensor<?x512xf32>) -> tensor<?x512xf32>
        %8 = scf.for %arg4 = %c0 to %c

In [100]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-insert-quantization{ops=gemm,gemv qtype=i8 scale=0.03125 zp=0 rounding=nearest narrow-range=false})"
    ")"
)

gemm_cinm6 = torch_nb.ARTIFACTS_DIR / "gemm_cinm6.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm5,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm6,
    ]
)

print(gemm_cinm5.read_text())


module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %0 = cinm.compute attributes {tileSizes = array<i64: 32, 32>} -> tensor<1024x512xf32> {
      %1 = tensor.empty() : tensor<1024x512xf32>
      %2 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %1) -> (tensor<1024x512xf32>) {
        %3 = arith.subi %c1024, %arg2 : index
        %4 = arith.cmpi ugt, %3, %c32 : index
        %5 = arith.select %4, %c32, %3 : index
        %cst_0 = arith.constant 0.000000e+00 : f32
        %6 = tensor.empty(%5) : tensor<?x512xf32>
        %7 = linalg.fill ins(%cst_0 : f32) outs(%6 : tensor<?x512xf32>) -> tensor<?x512xf32>
        %8 = scf.for %arg4 = %c0 to %c

In [102]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "lower-affine,"
    "func.func(cinm-relower)"
    ")"
)

gemm_cinm7 = torch_nb.ARTIFACTS_DIR / "gemm_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm6,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm7,
    ]
)

print(gemm_cinm7.read_text())

module {
  func.func @kernel(%arg0: tensor<1024x2048xf32>, %arg1: tensor<2048x512xf32>) -> tensor<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %0 = cinm.compute attributes {tileSizes = array<i64: 32, 32>} -> tensor<1024x512xf32> {
      %1 = tensor.empty() : tensor<1024x512xf32>
      %2 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %1) -> (tensor<1024x512xf32>) {
        %3 = arith.subi %c1024, %arg2 : index
        %4 = arith.cmpi ugt, %3, %c32 : index
        %5 = arith.select %4, %c32, %3 : index
        %cst_0 = arith.constant 0.000000e+00 : f32
        %6 = tensor.empty(%5) : tensor<?x512xf32>
        %7 = linalg.fill ins(%cst_0 : f32) outs(%6 : tensor<?x512xf32>) -> tensor<?x512xf32>
        %8 = scf.for %arg4 = %c0 to %c

In [107]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(linalg-generalize-named-ops,canonicalize,scf-for-loop-canonicalization),"
    "one-shot-bufferize{bufferize-function-boundaries}"
    ")"
)

gemm_cinm7 = torch_nb.ARTIFACTS_DIR / "gemm_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm6,  # produced in the previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm7,
    ]
)

print(gemm_cinm7.read_text())


#map = affine_map<(d0, d1) -> ()>
#map1 = affine_map<(d0, d1) -> (d0, d1)>
module {
  func.func @kernel(%arg0: memref<1024x2048xf32, strided<[?, ?], offset: ?>>, %arg1: memref<2048x512xf32, strided<[?, ?], offset: ?>>) -> memref<1024x512xf32> {
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %0 = cinm.compute_memref attributes {tileSizes = array<i64: 32, 32>} -> memref<1024x512xf32> {
      %alloc = memref.alloc() {alignment = 64 : i64} : memref<1024x512xf32>
      %1 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %alloc) -> (memref<1024x512xf32>) {
        %2 = arith.subi %c1024, %arg2 : index
        %3 = arith.cmpi ugt, %2, %c32 : index
        %4 = arith.select %3, %c32, %2 : index
        %alloc_0 = memref.alloc(%4) {alignment = 64 : i64} : me

In [110]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "cinm-memory-cleanup,"
    "func.func(convert-cinm-to-cim,cim-mark-relower{ops=add,relu})"
    ")"
)

gemm_cinm8 = torch_nb.ARTIFACTS_DIR / "gemm_cinm8.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm7,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm8,
    ]
)

print(gemm_cinm8.read_text())

#map = affine_map<(d0, d1) -> ()>
#map1 = affine_map<(d0, d1) -> (d0, d1)>
module {
  func.func @kernel(%arg0: memref<1024x2048xf32, strided<[?, ?], offset: ?>>, %arg1: memref<2048x512xf32, strided<[?, ?], offset: ?>>) -> memref<1024x512xf32> {
    %c0_i8 = arith.constant 0 : i8
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %cim_dev = cim.acquire_device -> !cim.deviceId
    %cim_cbr = cim.acquire_crossbar %cim_dev {height = 32 : i64, width = 32 : i64} : !cim.deviceId -> !cim.crossbarId
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<1024x512xf32>
    %0 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %alloc) -> (memref<1024x512xf32>) {
      %1 = arith.subi %c1024, %arg2 : index
      %2 = arith.cmpi ugt, %1, %c32 : index
      %3 = ar

In [111]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(convert-cim-to-alpine,cim-cleanup-unsupported),"
    "func.func(alpine-hoist-write-weights)"
    ")"
)

gemm_cinm9 = torch_nb.ARTIFACTS_DIR / "gemm_cinm9.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm8,  # input from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm9,
    ]
)

print(gemm_cinm9.read_text())

#map = affine_map<(d0, d1) -> ()>
#map1 = affine_map<(d0, d1) -> (d0, d1)>
module {
  func.func @kernel(%arg0: memref<1024x2048xf32, strided<[?, ?], offset: ?>>, %arg1: memref<2048x512xf32, strided<[?, ?], offset: ?>>) -> memref<1024x512xf32> {
    %c0_i8 = arith.constant 0 : i8
    %cst = arith.constant 0.000000e+00 : f32
    %c32 = arith.constant 32 : index
    %c2048 = arith.constant 2048 : index
    %c1024 = arith.constant 1024 : index
    %c1 = arith.constant 1 : index
    %c512 = arith.constant 512 : index
    %c0 = arith.constant 0 : index
    %alpine_tile = alpine.alloc_tile {height = 32 : i64, width = 32 : i64} -> i32
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<1024x512xf32>
    %0 = scf.for %arg2 = %c0 to %c1024 step %c32 iter_args(%arg3 = %alloc) -> (memref<1024x512xf32>) {
      %1 = arith.subi %c1024, %arg2 : index
      %2 = arith.cmpi ugt, %1, %c32 : index
      %3 = arith.select %2, %c32, %1 : index
      %alloc_0 = memref.alloc(%3) {alignment = 64 : i64

In [116]:
from tutorial._infra import cinm_frontend as cinm_nb
from tutorial._infra import torch_frontend as torch_nb

cinm_pipeline = (
    "builtin.module("
    "convert-alpine-to-func,"
    "convert-linalg-to-loops,lower-affine,convert-scf-to-cf,canonicalize,"
    "resolve-ranked-shaped-type-result-dims,expand-strided-metadata,"
    "memref-expand,canonicalize,lower-affine,canonicalize,"
    "convert-vector-to-llvm,convert-math-to-llvm,convert-arith-to-llvm,"
    "convert-index-to-llvm,convert-scf-to-cf,convert-cf-to-llvm,"
    "convert-func-to-llvm,finalize-memref-to-llvm,canonicalize,"
    "convert-to-llvm,reconcile-unrealized-casts,"
    "canonicalize"
    ")"
)

gemm_cinm9 = torch_nb.ARTIFACTS_DIR / "gemm_cinm9.mlir"
gemm_cinm10 = torch_nb.ARTIFACTS_DIR / "gemm_cinm10.mlir"

cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        gemm_cinm9,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        gemm_cinm10,
    ]
)

print(gemm_cinm10.read_text())

module {
  llvm.func @memrefCopy(i64, !llvm.ptr, !llvm.ptr)
  llvm.func @malloc(i64) -> !llvm.ptr
  llvm.func @alpine_alloc_tile(i64, i64) -> i32 attributes {sym_visibility = "nested"}
  llvm.func @alpine_quantize_r2(!llvm.ptr, !llvm.ptr, i64, i64, i64, i64, i64, !llvm.ptr, !llvm.ptr, i64, i64, i64, i64, i64, f32, i32) attributes {sym_visibility = "nested"}
  llvm.func @alpine_write_weights(i64, !llvm.ptr, !llvm.ptr, i64, i64, i64, i64, i64, i64) attributes {sym_visibility = "nested"}
  llvm.func @alpine_quantize_r1(!llvm.ptr, !llvm.ptr, i64, i64, i64, !llvm.ptr, !llvm.ptr, i64, i64, i64, f32, i32) attributes {sym_visibility = "nested"}
  llvm.func @alpine_enqueue_vec(i64, !llvm.ptr, !llvm.ptr, i64, i64, i64, i64) attributes {sym_visibility = "nested"}
  llvm.func @alpine_process(i32) attributes {sym_visibility = "nested"}
  llvm.func @alpine_dequeue_vec(i64, !llvm.ptr, !llvm.ptr, i64, i64, i64, i64) attributes {sym_visibility = "nested"}
  llvm.func @alpine_dequantize_r1(!llvm.ptr, !l